In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import json
import numpy as np
import torch
import matplotlib.pyplot as plt
from phantominator import shepp_logan

from DcTNN.model import cascadeNet, axVIT, patchVIT
from DcTNN.dc import FFT_DC, KSpace_DC, fft_2d, ifft_2d
from dataset import MRIDataset, load_mask

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
CHECKPOINT_BASE = '../Experiments'
DATA_DIR        = '/scratch/user/uqanag/OASIS/keras_png_slices_train'
MASK_DIR        = '../masks'
N               = 320
ACCEL           = 8

In [ ]:
def build_model_from_cfg(cfg):
    k_space  = cfg.get('k_space_learning', False)
    encoders = cfg.get('encoders', ['patch', 'patch', 'patch'])

    enc_list, enc_args = [], []
    for name in encoders:
        ch = 2 if k_space else cfg.get('num_channels', 1)
        if name == 'axial':
            enc_list.append(axVIT)
            enc_args.append(dict(
                layerNo=cfg.get('layer_no', 1), numCh=ch, d_model=None,
                nhead=cfg.get('nhead_axial', 8),
                num_encoder_layers=cfg.get('num_encoder_layers', 2),
                dim_feedforward=None, pos_emb_type=cfg.get('pos_emb_type', 'APE'),
                rope_theta=cfg.get('rope_theta', 100.0),
                rope_mixed_rotate=cfg.get('rope_mixed_rotate', True),
            ))
        else:
            enc_list.append(patchVIT)
            enc_args.append(dict(
                patch_size=cfg.get('patch_size', 16),
                kaleidoscope=(name == 'kaleidoscope'),
                layerNo=cfg.get('layer_no', 1), numCh=ch,
                nhead=cfg.get('nhead_patch', 8),
                num_encoder_layers=cfg.get('num_encoder_layers', 2),
                dim_feedforward=None, d_model=None,
                pos_emb_type=cfg.get('pos_emb_type', 'APE'),
                rope_theta=cfg.get('rope_theta', 100.0),
                rope_mixed_rotate=cfg.get('rope_mixed_rotate', True),
            ))

    use_lamb = (cfg.get('lambda_schedule', 'none') == 'none')
    return cascadeNet(cfg.get('image_size', 320), enc_list, enc_args,
                      use_lamb, k_space_learning=k_space)


def load_experiment(name):
    exp_dir     = os.path.join(CHECKPOINT_BASE, name)
    config_path = os.path.join(exp_dir, 'config.json')
    ckpt_path   = os.path.join(exp_dir, 'best_model.pth')

    with open(config_path) as f:
        cfg = json.load(f)

    model = build_model_from_cfg(cfg).to(device)

    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ckpt['model'])
        ep   = ckpt.get('epoch', '?')
        psnr = ckpt.get('val_psnr', float('nan'))
        print(f'  {name:45s}  epoch={ep}, val_psnr={psnr:.2f} dB')
    else:
        print(f'  {name:45s}  WARNING — no checkpoint, using random weights')

    model.eval()
    return model


EXP_NAMES = [
    'KSpace_patch',
    'KSpace_kaleidoscope',
    'KSpace_axial',
    'Lambda_Schedule_patch_cosine',
]
MODEL_LABELS = {
    'KSpace_patch':                 'KSpace Patch',
    'KSpace_kaleidoscope':          'KSpace Kaleidoscope',
    'KSpace_axial':                 'KSpace Axial',
    'Lambda_Schedule_patch_cosine': 'Baseline (Image Space)',
}

print('Loading models...')
models = {name: load_experiment(name) for name in EXP_NAMES}
print('Done.')

In [ ]:
# ── Inputs ─────────────────────────────────────────────────────────────────────
ph_np     = np.rot90(np.transpose(np.array(shepp_logan(N))), 1).astype(np.float32)
ph_tensor = torch.tensor(ph_np.copy()).unsqueeze(0).unsqueeze(0)  # [1,1,N,N]

dataset  = MRIDataset(DATA_DIR, N=N, split='val', val_fraction=0.1, seed=42)
real_img = dataset[0].unsqueeze(0)  # [1,1,N,N]

mask = load_mask(os.path.join(MASK_DIR, f'mask_R{ACCEL}.png'), N).to(device)


def simulate_undersampling(gt):
    gt    = gt.to(device)
    ks_f  = fft_2d(gt)
    ks_us = ks_f * mask
    zf    = ifft_2d(ks_us)[:, 0:1, :, :]
    return zf, ks_us


zf_phantom, ks_phantom = simulate_undersampling(ph_tensor)
zf_real,    ks_real    = simulate_undersampling(real_img)
gt_phantom = ph_tensor.to(device)
gt_real    = real_img.to(device)

In [ ]:
# ── Display helpers ────────────────────────────────────────────────────────────
def to_img(t):
    return np.abs(t[0, 0].cpu().numpy())


def to_ks(t):
    ks = fft_2d(t.to(device))
    r  = ks[0, 0].cpu().numpy()
    im = ks[0, 1].cpu().numpy()
    return np.log(np.fft.fftshift(np.sqrt(r**2 + im**2)) + 1e-8)


def mse_img(gt, recon):
    return (to_img(gt) - to_img(recon)) ** 2


def mse_ks(gt, recon):
    gt_ks    = fft_2d(gt.to(device))
    recon_ks = fft_2d(recon.to(device))
    diff_r   = (gt_ks[0, 0] - recon_ks[0, 0]).cpu().numpy()
    diff_i   = (gt_ks[0, 1] - recon_ks[0, 1]).cpu().numpy()
    return np.fft.fftshift(diff_r**2 + diff_i**2)


def calc_psnr(gt_im, recon_im, max_val=None):
    if max_val is None:
        max_val = gt_im.max()
    mse = np.mean((gt_im - recon_im) ** 2)
    return 20 * np.log10(max_val / (np.sqrt(mse) + 1e-12))


# ── Inference + plot ───────────────────────────────────────────────────────────
def run_and_plot(gt, zf, y, input_label):
    n_rows = len(EXP_NAMES) + 1          # +1 for the reference row at top
    fig, axes = plt.subplots(n_rows, 4, figsize=(16, 3.8 * n_rows))
    fig.suptitle(
        f'Inference Results — {input_label}  (R={ACCEL})',
        fontsize=13, fontweight='bold', y=1.01
    )

    gt_im    = to_img(gt)
    gt_ks    = to_ks(gt)
    zf_im    = to_img(zf)
    zf_ks    = to_ks(zf)
    max_val  = gt_im.max()
    vmax_img = np.percentile(gt_im, 99)
    vmin_ks, vmax_ks = np.percentile(gt_ks, 1), np.percentile(gt_ks, 99)

    # ── Row 0: reference ──────────────────────────────────────────────────────
    ref_data   = [gt_im,  gt_ks,  zf_im,  zf_ks]
    ref_cmaps  = ['gray', 'inferno', 'gray', 'inferno']
    ref_titles = ['GT Image', 'GT K-space', 'Undersampled Image', 'Undersampled K-space']
    ref_vlims  = [(0, vmax_img), (vmin_ks, vmax_ks), (0, vmax_img), (vmin_ks, vmax_ks)]

    for col, (data, cmap, title, (vmin, vmax)) in enumerate(
            zip(ref_data, ref_cmaps, ref_titles, ref_vlims)):
        axes[0, col].imshow(data, cmap=cmap, vmin=vmin, vmax=vmax)
        axes[0, col].set_title(title, fontsize=9, fontweight='bold')
        axes[0, col].axis('off')
    axes[0, 0].text(
        -0.04, 0.5, 'Reference', transform=axes[0, 0].transAxes,
        fontsize=9, fontweight='bold', va='center', ha='right', rotation=90,
    )

    # ── Column titles for model rows ──────────────────────────────────────────
    for col, t in enumerate(['Recon Image', 'Recon K-space', 'MSE — Image', 'MSE — K-space']):
        axes[1, col].set_title(t, fontsize=9, fontweight='bold')

    # ── Rows 1–N: one per model ───────────────────────────────────────────────
    print(f'\n--- PSNR  ({input_label}, R={ACCEL}) ---')
    for i, name in enumerate(EXP_NAMES):
        row = i + 1
        with torch.no_grad():
            recon = models[name](zf.to(device), y.to(device), mask).cpu()

        rc_im  = to_img(recon)
        rc_ks  = to_ks(recon)
        err_im = mse_img(gt, recon)
        err_ks = mse_ks(gt, recon)
        psnr   = calc_psnr(gt_im, rc_im, max_val)

        print(f'  {MODEL_LABELS[name]:30s}  PSNR = {psnr:.2f} dB')

        axes[row, 0].imshow(rc_im,  cmap='gray',    vmin=0,       vmax=vmax_img)
        axes[row, 1].imshow(rc_ks,  cmap='inferno', vmin=vmin_ks, vmax=vmax_ks)
        axes[row, 2].imshow(err_im, cmap='hot',     vmin=0,       vmax=np.percentile(err_im, 99))
        axes[row, 3].imshow(np.log(err_ks + 1e-8),  cmap='hot')

        for col in range(4):
            axes[row, col].axis('off')
        axes[row, 0].text(
            -0.04, 0.5, f"{MODEL_LABELS[name]}\nPSNR {psnr:.1f} dB",
            transform=axes[row, 0].transAxes,
            fontsize=8, fontweight='bold', va='center', ha='right', rotation=90,
        )

    plt.tight_layout()
    plt.show()


run_and_plot(gt_phantom, zf_phantom, ks_phantom, 'Shepp-Logan Phantom')
run_and_plot(gt_real,    zf_real,    ks_real,    'Real OASIS Image')

In [ ]:

# ── Global MSE Correlation ─────────────────────────────────────────────────────
# Computes Pearson r between per-sample scalar MSE of two models.
# A high r means the models fail (and succeed) on the same samples.

def mse_correlation(name_a, name_b, n_samples=50):
    """
    Returns the Pearson correlation between per-sample MSE of name_a and name_b.
    Runs inference on n_samples from the validation set.
    """
    model_a = models[name_a]
    model_b = models[name_b]
    label_a = MODEL_LABELS.get(name_a, name_a)
    label_b = MODEL_LABELS.get(name_b, name_b)

    mses_a, mses_b = [], []

    for i in range(n_samples):
        gt_raw      = dataset[i].unsqueeze(0)           # [1,1,N,N]
        zf_i, y_i   = simulate_undersampling(gt_raw)
        gt_np       = to_img(gt_raw)

        with torch.no_grad():
            recon_a = model_a(zf_i.to(device), y_i.to(device), mask).cpu()
            recon_b = model_b(zf_i.to(device), y_i.to(device), mask).cpu()

        mses_a.append(np.mean((gt_np - to_img(recon_a)) ** 2))
        mses_b.append(np.mean((gt_np - to_img(recon_b)) ** 2))

    mses_a = np.array(mses_a)
    mses_b = np.array(mses_b)
    r      = np.corrcoef(mses_a, mses_b)[0, 1]

    print(f"Global MSE Correlation — {label_a}  vs  {label_b}")
    print(f"  n_samples  : {n_samples}")
    print(f"  mean MSE A : {mses_a.mean():.6f}")
    print(f"  mean MSE B : {mses_b.mean():.6f}")
    print(f"  Pearson r  : {r:.4f}")

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(mses_a, mses_b, alpha=0.6, s=20, color='steelblue')
    m, b = np.polyfit(mses_a, mses_b, 1)
    xs   = np.linspace(mses_a.min(), mses_a.max(), 200)
    ax.plot(xs, m * xs + b, 'r--', lw=1.5, label=f'r = {r:.4f}')
    ax.set_xlabel(f'MSE — {label_a}')
    ax.set_ylabel(f'MSE — {label_b}')
    ax.set_title(f'Per-sample MSE Correlation\n{label_a} vs {label_b}')
    ax.legend()
    plt.tight_layout()
    plt.show()

    return r


# Example usage — swap names as needed
# r = mse_correlation('KSpace_patch', 'KSpace_kaleidoscope', n_samples=50)


In [ ]:

# ── Local MSE Correlation Heatmap ──────────────────────────────────────────────
# Row 1: image-space mean MSE + patch correlation
# Row 2: k-space  mean MSE + patch correlation

def local_mse_correlation(name_a, name_b, patch_size=16, n_samples=30):
    """
    Returns (corr_map_img, corr_map_ks).
    Rows: image space (top) | k-space (bottom).
    Columns: mean MSE A | mean MSE B | patch correlation heatmap.
    """
    model_a = models[name_a]
    model_b = models[name_b]
    label_a = MODEL_LABELS.get(name_a, name_a)
    label_b = MODEL_LABELS.get(name_b, name_b)

    mse_img_a, mse_img_b = [], []
    mse_ks_a,  mse_ks_b  = [], []

    for i in range(n_samples):
        gt_raw    = dataset[i].unsqueeze(0)
        zf_i, y_i = simulate_undersampling(gt_raw)
        gt_np     = to_img(gt_raw)

        with torch.no_grad():
            recon_a = model_a(zf_i.to(device), y_i.to(device), mask).cpu()
            recon_b = model_b(zf_i.to(device), y_i.to(device), mask).cpu()

        mse_img_a.append((gt_np - to_img(recon_a)) ** 2)
        mse_img_b.append((gt_np - to_img(recon_b)) ** 2)
        mse_ks_a.append(mse_ks(gt_raw, recon_a))
        mse_ks_b.append(mse_ks(gt_raw, recon_b))

    mse_img_a = np.stack(mse_img_a)
    mse_img_b = np.stack(mse_img_b)
    mse_ks_a  = np.stack(mse_ks_a)
    mse_ks_b  = np.stack(mse_ks_b)

    def patch_corr(maps_a, maps_b):
        H, W = maps_a.shape[1], maps_a.shape[2]
        n_ph, n_pw = H // patch_size, W // patch_size
        cmap = np.zeros((n_ph, n_pw))
        for pi in range(n_ph):
            for pj in range(n_pw):
                r0, r1 = pi * patch_size, (pi + 1) * patch_size
                c0, c1 = pj * patch_size, (pj + 1) * patch_size
                pa = maps_a[:, r0:r1, c0:c1].reshape(n_samples, -1).mean(axis=1)
                pb = maps_b[:, r0:r1, c0:c1].reshape(n_samples, -1).mean(axis=1)
                cmap[pi, pj] = np.corrcoef(pa, pb)[0, 1]
        return cmap

    corr_img = patch_corr(mse_img_a, mse_img_b)
    corr_ks  = patch_corr(mse_ks_a,  mse_ks_b)

    mean_img_a = mse_img_a.mean(axis=0)
    mean_img_b = mse_img_b.mean(axis=0)
    mean_ks_a  = np.log(mse_ks_a.mean(axis=0) + 1e-8)
    mean_ks_b  = np.log(mse_ks_b.mean(axis=0) + 1e-8)

    vmax_img = np.percentile(np.stack([mean_img_a, mean_img_b]), 99)
    vmin_ks  = np.percentile(np.stack([mean_ks_a,  mean_ks_b]),  1)
    vmax_ks  = np.percentile(np.stack([mean_ks_a,  mean_ks_b]),  99)

    r_img = np.corrcoef(mse_img_a.mean(axis=(1,2)), mse_img_b.mean(axis=(1,2)))[0, 1]
    r_ks  = np.corrcoef(mse_ks_a.mean(axis=(1,2)),  mse_ks_b.mean(axis=(1,2)))[0, 1]
    med_r_img = np.nanmedian(corr_img)
    med_r_ks  = np.nanmedian(corr_ks)

    def annotate(ax, text):
        ax.text(0.5, 0.02, text, transform=ax.transAxes,
                fontsize=8, color='white', ha='center', va='bottom',
                bbox=dict(boxstyle='round,pad=0.2', fc='black', alpha=0.55))

    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    fig.suptitle(
        f'Local MSE Correlation: {label_a}  vs  {label_b}'
        f'\npatch={patch_size}px  |  n={n_samples}',
        fontweight='bold'
    )

    # ── Row 0: image space ────────────────────────────────────────────────────
    axes[0, 0].imshow(mean_img_a, cmap='hot', vmin=0, vmax=vmax_img)
    axes[0, 0].set_title(f'Mean MSE (image)\n{label_a}', fontsize=9)
    annotate(axes[0, 0], f'mean MSE = {mean_img_a.mean():.2e}')

    axes[0, 1].imshow(mean_img_b, cmap='hot', vmin=0, vmax=vmax_img)
    axes[0, 1].set_title(f'Mean MSE (image)\n{label_b}', fontsize=9)
    annotate(axes[0, 1], f'mean MSE = {mean_img_b.mean():.2e}')

    im0 = axes[0, 2].imshow(corr_img, cmap='RdYlGn', vmin=-1, vmax=1, interpolation='nearest')
    axes[0, 2].set_title('Patch Correlation (image space)', fontsize=9)
    plt.colorbar(im0, ax=axes[0, 2], fraction=0.046, pad=0.04, label='Pearson r')
    annotate(axes[0, 2], f'median patch r = {med_r_img:.4f}  |  global r = {r_img:.4f}')

    # ── Row 1: k-space ────────────────────────────────────────────────────────
    axes[1, 0].imshow(mean_ks_a, cmap='inferno', vmin=vmin_ks, vmax=vmax_ks)
    axes[1, 0].set_title(f'Mean MSE K-space (log)\n{label_a}', fontsize=9)
    annotate(axes[1, 0], f'mean log-MSE = {mean_ks_a.mean():.2f}')

    axes[1, 1].imshow(mean_ks_b, cmap='inferno', vmin=vmin_ks, vmax=vmax_ks)
    axes[1, 1].set_title(f'Mean MSE K-space (log)\n{label_b}', fontsize=9)
    annotate(axes[1, 1], f'mean log-MSE = {mean_ks_b.mean():.2f}')

    im1 = axes[1, 2].imshow(corr_ks, cmap='RdYlGn', vmin=-1, vmax=1, interpolation='nearest')
    axes[1, 2].set_title('Patch Correlation (k-space)', fontsize=9)
    plt.colorbar(im1, ax=axes[1, 2], fraction=0.046, pad=0.04, label='Pearson r')
    annotate(axes[1, 2], f'median patch r = {med_r_ks:.4f}  |  global r = {r_ks:.4f}')

    for ax in axes.flat:
        ax.axis('off')

    plt.tight_layout()
    plt.show()

    return corr_img, corr_ks


